# Lab: a single-SNP association test

In this lab you will run the same association test three ways:

1. **Python**: count alleles and compute an odds ratio and a χ² P-value by hand.
2. **R**: fit a logistic regression and see that $e^{\beta_1}$ is the odds ratio.
3. **PLINK**: run an association test across 100 SNPs, the way a real GWAS does.

The data are simulated: 1,000 cases and 1,000 controls genotyped at 100 SNPs, one of which (`rs_causal`)
truly raises disease risk.

See *Running the lab notebooks* in the book for where to run this notebook.

## Setup

In [ ]:
import shutil
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

rng = np.random.default_rng(17)
for tool in ["plink", "R"]:
    print(f"{tool:6s}", "found" if shutil.which(tool) else "NOT found (that part of the lab will not run)")

## Simulate the study

Each SNP has a control allele frequency between 0.1 and 0.9. For the causal SNP we raise the
frequency of the C allele in cases so that its odds ratio is about 1.7, as in the lecture example.

In [ ]:
n_cases, n_controls, n_snps = 1000, 1000, 100
causal = 42
f_controls = rng.uniform(0.1, 0.9, n_snps)
f_cases = f_controls.copy()
odds_cases = 1.7 * f_controls[causal] / (1 - f_controls[causal])     # OR = 1.7 at the causal SNP
f_cases[causal] = odds_cases / (1 + odds_cases)

# genotype = number of C alleles (0, 1 or 2), drawn from each group's allele frequency
G = np.vstack([rng.binomial(2, f_cases, (n_cases, n_snps)),
               rng.binomial(2, f_controls, (n_controls, n_snps))])
status = np.r_[np.ones(n_cases, int), np.zeros(n_controls, int)]   # 1 = case, 0 = control
snp_ids = [f"rs{1000 + j}" for j in range(n_snps)]
snp_ids[causal] = "rs_causal"
print(G.shape, "genotype matrix (people x SNPs)")

## Part 1 — Python: odds ratio and χ² by hand

**Exercise 1.** Complete `allele_table` so it returns a 2 × 2 table of allele counts
(rows: cases, controls; columns: C, T). Hint: each person contributes two alleles.

In [ ]:
def allele_table(g, status):
    """g: genotypes (number of C alleles) at one SNP; status: 1 = case, 0 = control."""
    c_cases = g[status == 1].sum()
    t_cases = 2 * (status == 1).sum() - c_cases
    c_controls = g[status == 0].sum()
    t_controls = 2 * (status == 0).sum() - c_controls
    return np.array([[c_cases, t_cases], [c_controls, t_controls]])

tab = allele_table(G[:, causal], status)
odds_ratio = (tab[0, 0] / tab[0, 1]) / (tab[1, 0] / tab[1, 1])
chi2, p, dof, _ = chi2_contingency(tab, correction=False)
print(pd.DataFrame(tab, index=["cases", "controls"], columns=["C", "T"]))
print(f"\nOR = {odds_ratio:.2f}   chi2 = {chi2:.1f}   P = {p:.1e}")

**Exercise 2.** Run the same test on every SNP and collect the results in a table.
How many SNPs pass P < 0.05? How many would you expect by chance alone among 99 null SNPs?

In [ ]:
rows = []
for j, sid in enumerate(snp_ids):
    t = allele_table(G[:, j], status)
    chi2, p, *_ = chi2_contingency(t, correction=False)
    rows.append((sid, (t[0, 0] / t[0, 1]) / (t[1, 0] / t[1, 1]), chi2, p))
res = pd.DataFrame(rows, columns=["SNP", "OR", "chi2", "P"]).sort_values("P")
print("SNPs with P < 0.05:", (res.P < 0.05).sum())
res.head()

## Part 2 — R: logistic regression

The `%%R` cells run R inside this notebook (via `rpy2`); `-i` copies a Python variable into R.
In a logistic regression of case status on allele dosage, the slope $\beta_1$ is the log odds ratio
per C allele, so $e^{\beta_1}$ should be close to the OR from Part 1.

In [ ]:
%load_ext rpy2.ipython
dosage = G[:, causal]

In [ ]:
%%R -i dosage,status
fit <- glm(status ~ dosage, family = binomial)
print(summary(fit)$coefficients)
cat("\nOR per C allele = exp(beta1) =", round(exp(coef(fit)["dosage"]), 2), "\n")

**Exercise 3.** Add a covariate. Simulate an `age` variable that differs between cases and controls,
add it to the model (`status ~ dosage + age`) and compare the OR for `dosage` with and without it.
Why does it barely change here? (Hint: is age related to genotype in this simulation?)

## Part 3 — PLINK: a mini GWAS

PLINK reads genotypes in `.ped`/`.map` text format. We write our simulated data in that format, then run
PLINK's allelic association test (`--assoc`), which is the same χ² test you did in Part 1.

In [ ]:
geno_str = np.where(G == 2, "C C", np.where(G == 1, "C T", "T T"))
with open("sim.ped", "w") as fh:
    for i in range(G.shape[0]):
        pheno = 2 if status[i] == 1 else 1          # PLINK: 2 = case, 1 = control
        fh.write(f"F{i} I{i} 0 0 0 {pheno} " + " ".join(geno_str[i]) + "\n")
with open("sim.map", "w") as fh:
    for j, sid in enumerate(snp_ids):
        fh.write(f"1 {sid} 0 {1_000_000 + 10_000 * j}\n")
print("wrote sim.ped and sim.map")

In [ ]:
%%bash
plink --file sim --assoc --allow-no-sex --out sim_assoc > /dev/null
sort -g -k9,9 sim_assoc.assoc | head -6

**Exercise 4.** Compare PLINK's `OR` and `P` for `rs_causal` with your Part 1 results.
Then check: is PLINK's `A1` allele C or T for that SNP, and how does that affect the OR it reports?

**Exercise 5.** Make a Manhattan-style plot of $-\log_{10}(P)$ against SNP position for the 100 SNPs,
with a dashed line at P = 0.05 / 100 (a Bonferroni threshold). Which SNPs pass?